In [ ]:
import os
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from plot_style import apply_publication_style
apply_publication_style()
import numpy as np
import seaborn as sns
from scipy import stats

# Exploratory data analysis: HH→bbγγ

All SM and BSM SMEFT datasets (March 2026 HDF5). Standard pipeline plots are written to `plots/`.

To regenerate without this notebook: `python reproduce.py eda`.


### 1. Load data

In [ ]:
def load_dataset(filepath):
    with h5py.File(filepath, 'r') as f:
        df = pd.DataFrame.from_records(f['ForAnalysis/1d'][:])
    return df

sm_df = load_dataset('datasets/new_Input_bbyy_SMEFT_SM_4thMarch_2026.h5')
cbbim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_cbbim_4thMarch_2026.h5')
cbgim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_cbgim_4thMarch_2026.h5')
cbhim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_cbhim_4thMarch_2026.h5')
chbtil_df = load_dataset('datasets/new_Input_bbyy_SMEFT_chbtil_4thMarch_2026.h5')
chgtil_df = load_dataset('datasets/new_Input_bbyy_SMEFT_chgtil_4thMarch_2026.h5')
ctbim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_ctbim_4thMarch_2026.h5')

for df in [sm_df, cbbim_df, cbgim_df, cbhim_df, chbtil_df, chgtil_df, ctbim_df]:
    df['is_HiggsEvent'] = df['is_HHEvent'] | df['is_SingleHiggsEvent'] | df['is_ZHEvent']

datasets = {
    'SM': sm_df, 'cbbim': cbbim_df, 'cbgim': cbgim_df,
    'cbhim': cbhim_df, 'chbtil': chbtil_df, 'chgtil': chgtil_df, 'ctbim': ctbim_df
}
colors = {'SM': 'blue', 'cbbim': 'orange', 'cbgim': 'red', 'cbhim': 'green',
          'chbtil': 'purple', 'chgtil': 'brown', 'ctbim': 'magenta'}

os.makedirs('plots', exist_ok=True)

for name, df in datasets.items():
    print(f"{name}: {len(df):,} events")
print(f"\nColumns ({len(sm_df.columns)}): {list(sm_df.columns)}")

In [ ]:
# Summary statistics: df.describe() — omitted to avoid verbose output

In [ ]:
# All 32 physics observables (exclude metadata, event-type flags, and boolean columns)
exclude_columns = ['EventNumber', 'is_HHEvent', 'is_SingleHiggsEvent', 'is_SingleZEvent', 'is_ZHEvent', 'is_HiggsEvent', 'Lumi_weight', 'nBTaggedJets', 'NJets']
observables = [c for c in sm_df.columns if c not in exclude_columns]
key_observables = ['Higgs_Mass', 'Higgs_pT', 'LeadPhoton_pT', 'pT_jj', 'SubLeadJet_pT', 'DPhi_bb', 'm_bbyy', 'M_jj', 'LeadJet_pT', 'Higgs_Eta', 'signed_DeltaPhi_jj']

### 2. Distribution comparison (all observables)

In [ ]:
n_obs = len(observables)
n_cols = 4
n_rows = (n_obs + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes = axes.flatten()

for idx, obs in enumerate(observables):
    ax = axes[idx]
    all_data = np.concatenate([df[obs].dropna().values for df in datasets.values()])
    bins = np.linspace(np.percentile(all_data, 1), np.percentile(all_data, 99), 50)
    for name, df in datasets.items():
        ax.hist(df[obs].dropna(), bins=bins, density=True,
                label=name, color=colors[name], histtype='step', linewidth=2)
    ax.set_xlabel(obs)
    ax.set_ylabel('Normalized Density') if idx % n_cols == 0 else ax.set_ylabel('')
    if idx % n_cols != 0:
        ax.tick_params(axis='y', labelleft=False)
    if idx % n_cols == 0:
        ax.legend(fontsize=15)
    ax.set_title(f'{obs}')

for idx in range(n_obs, len(axes)):
    axes[idx].set_visible(False)
plt.tight_layout()
plt.savefig('plots/dataset_comparison_all.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(32, 14))
axes = axes.flatten()
for idx, (name, df) in enumerate(datasets.items()):
    if idx < len(axes):
        corr = df[observables].corr()
        sns.heatmap(corr, ax=axes[idx], cmap='RdBu_r', center=0, vmin=-1, vmax=1, annot=False, square=True)
        axes[idx].set_title(f'{name}', fontsize=12)
        axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=45, ha='right', fontsize=6)
        axes[idx].set_yticklabels(axes[idx].get_yticklabels(), fontsize=6)
if len(datasets) < len(axes):
    axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig('plots/correlation_matrices_all.png', dpi=150, bbox_inches='tight')
plt.show()

### 3. Correlation matrices & 2D histograms

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(28, 10))
axes = axes.flatten()
all_eta = np.concatenate([df['Eta_jj'].values for df in datasets.values()])
all_dphi = np.concatenate([df['signed_DeltaPhi_jj'].values for df in datasets.values()])
eta_bins = np.linspace(np.percentile(all_eta, 1), np.percentile(all_eta, 99), 50)
dphi_bins = np.linspace(np.percentile(all_dphi, 1), np.percentile(all_dphi, 99), 50)

for idx, (name, df) in enumerate(datasets.items()):
    ax = axes[idx]
    h, xedges, yedges = np.histogram2d(df['Eta_jj'], df['signed_DeltaPhi_jj'], bins=[eta_bins, dphi_bins])
    h_norm = h / h.sum()
    im = ax.pcolormesh(xedges, yedges, h_norm.T, cmap='viridis', shading='auto')
    ax.set_xlabel(r'$\eta_{jj}$')
    ax.set_ylabel(r'signed $\Delta\phi_{jj}$')
    ax.set_title(name)
    plt.colorbar(im, ax=ax)
if len(datasets) < len(axes):
    axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig('plots/2d_deltaphi_eta_all.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
bsm_names = [n for n in datasets.keys() if n != 'SM']
# Distribution overlap (lower = larger difference between SM and BSM)
def compute_histogram_overlap(vals1, vals2, bins=50):
    combined = np.concatenate([vals1, vals2])
    be = np.linspace(np.percentile(combined, 0.5), np.percentile(combined, 99.5), bins + 1)
    h1, _ = np.histogram(vals1, bins=be)
    h2, _ = np.histogram(vals2, bins=be)
    p1, p2 = h1 / (h1.sum() + 1e-10), h2 / (h2.sum() + 1e-10)
    return np.minimum(p1, p2).sum()

overlap_results = {}
for bsm in bsm_names:
    overlap_results[bsm] = {}
    for obs in observables:
        v1, v2 = datasets[bsm][obs].dropna().values, sm_df[obs].dropna().values
        overlap_results[bsm][obs] = compute_histogram_overlap(v1, v2) if len(v1) >= 10 and len(v2) >= 10 else np.nan

overlap_df = pd.DataFrame(overlap_results)

fig, ax = plt.subplots(figsize=(10, 14))
sns.heatmap(overlap_df, annot=False, cmap='RdYlGn', ax=ax, vmin=0.5, vmax=1.0,
            linewidths=0.5, linecolor='white')
ax.set_title('Distribution overlap: SM vs BSM (higher = more similar shapes)', fontsize=14)
ax.set_xlabel('BSM operator', fontsize=11)
ax.set_ylabel('Observable', fontsize=11)
plt.xticks(rotation=0)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('plots/distribution_overlap_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Top 5 discriminating observables per BSM (lowest overlap = most discriminating)
print('\nTop 5 discriminating observables per BSM vs SM:\n')
for col in overlap_df.columns:
    ranked = overlap_df[col].dropna().sort_values(ascending=True).head(5)
    print(f'{col}:')
    for obs, ov in ranked.items():
        print(f'  {obs}: overlap={ov:.3f}')
    print()


### 4. Distribution overlap & top discriminating observables (SM vs BSM)

In [ ]:
print("Event type breakdown:")
for name, df in datasets.items():
    n = len(df)
    hh = df['is_HHEvent'].sum()
    sh = df['is_SingleHiggsEvent'].sum()
    sz = df['is_SingleZEvent'].sum()
    zh = df['is_ZHEvent'].sum()
    other = n - hh - sh - sz - zh
    print(f"\n{name} ({n:,} total):")
    print(f"  HH: {hh:,} ({100*hh/n:.1f}%)")
    print(f"  Single Higgs: {sh:,} ({100*sh/n:.1f}%)")
    print(f"  Single Z: {sz:,} ({100*sz/n:.1f}%)")
    print(f"  ZH: {zh:,} ({100*zh/n:.1f}%)")
    print(f"  Other: {other:,} ({100*other/n:.1f}%)")

In [ ]:
# Top discriminating observables (from overlap heatmap above)
top_overlap = overlap_df.min(axis=1).sort_values(ascending=True)
top_features = list(top_overlap.head(5).index)

fig, axes = plt.subplots(1, len(top_features), figsize=(18, 4))

for idx, obs in enumerate(top_features):
    ax = axes[idx]
    all_data = np.concatenate([df[obs].dropna().values for df in datasets.values()])
    bins = np.linspace(np.percentile(all_data, 1), np.percentile(all_data, 99), 50)
    
    for name, df in datasets.items():
        ax.hist(df[obs].dropna(), bins=bins, density=True,
                label=name, color=colors[name], histtype='step', linewidth=2)
    
    ax.set_xlabel(obs, fontsize=11)
    ax.set_ylabel('Normalised Density' if idx == 0 else '')
    ax.set_title(obs, fontsize=11)
    ax.legend(fontsize=7)

plt.suptitle('SM vs BSM: Most Discriminating Observables', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('plots/top_features_comparison.png', dpi=150, bbox_inches='tight')


In [ ]:
feature_groups = {
    'CP-Sensitive Angular': [
        'signed_DeltaPhi_jj', 'cosThetaStar', 
        'costheta1', 'costheta2', 'DPhi_bb', 'DPhi_yybb'
    ],
    'Invariant Masses': [
        'm_bbyy', 'Higgs_Mass', 'M_jj', 'LeadJet_M', 'SubLeadJet_M'
    ],
    'Transverse Momenta': [
        'Higgs_pT', 'LeadJet_pT', 'SubLeadJet_pT', 
        'LeadPhoton_pT', 'SubLeadPhoton_pT', 'pT_jj'
    ]
}

for group_name, features in feature_groups.items():
    n_cols = 3
    n_rows = (len(features) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = axes.flatten()

    for idx, obs in enumerate(features):
        ax = axes[idx]
        all_data = np.concatenate([df[obs].dropna().values for df in datasets.values()])
        bins = np.linspace(np.percentile(all_data, 1), np.percentile(all_data, 99), 50)

        for name, df in datasets.items():
            ax.hist(df[obs].dropna(), bins=bins, density=True,
                    label=name, color=colors[name], histtype='step', linewidth=2)

        ax.set_xlabel(obs, fontsize=10)
        ax.set_ylabel('Normalised Density' if idx % n_cols == 0 else '')
        ax.legend(fontsize=6)

    for idx in range(len(features), len(axes)):
        axes[idx].set_visible(False)

    safe_name = group_name.lower().replace(' ', '_').replace('-', '')
    plt.suptitle(f'SM vs BSM: {group_name} Observables', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(f'plots/comparison_{safe_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# BSM/SM ratio panel: all 6 operators, extended observables
ratio_operators = bsm_names  # all 6: cbbim, cbgim, cbhim, chbtil, chgtil, ctbim

ratio_obs = [
    'signed_DeltaPhi_jj', 'DPhi_bb', 'costheta1', 'costheta2',
    'm_bbyy', 'M_jj', 'pT_jj', 'Eta_jj', 'Phi1'
]

obs_labels = {
    'signed_DeltaPhi_jj': r'$\Delta\tilde{\phi}_{jj}$',
    'DPhi_bb':            r'$\Delta\phi_{bb}$',
    'costheta1':          r'$\cos\theta_1$',
    'costheta2':          r'$\cos\theta_2$',
    'm_bbyy':             r'$m_{bb\gamma\gamma}$',
    'M_jj':               r'$M_{jj}$',
    'pT_jj':              r'$p_T^{jj}$',
    'Eta_jj':             r'$\eta_{jj}$',
    'Phi1':               r'$\Phi_1$',
}

sm_df = datasets['SM']
n_cols = 3
n_rows = (len(ratio_obs) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for idx, obs in enumerate(ratio_obs):
    ax = axes[idx]

    sm_vals = sm_df[obs].dropna().values
    all_vals = np.concatenate(
        [datasets[op][obs].dropna().values for op in ratio_operators] + [sm_vals]
    )
    bins = np.linspace(np.percentile(all_vals, 1), np.percentile(all_vals, 99), 40)
    bin_centres = 0.5 * (bins[:-1] + bins[1:])

    sm_counts, _ = np.histogram(sm_vals, bins=bins, density=True)
    sm_counts = np.where(sm_counts == 0, np.nan, sm_counts)

    # ±10% reference band
    ax.axhspan(0.90, 1.10, color='grey', alpha=0.15, label=r'$\pm10\%$')
    ax.axhline(1.0, color='black', linewidth=0.8, linestyle='--')

    for op in ratio_operators:
        bsm_vals = datasets[op][obs].dropna().values
        bsm_counts, _ = np.histogram(bsm_vals, bins=bins, density=True)
        ratio = bsm_counts / sm_counts

        ax.step(bin_centres, ratio, where='mid',
                color=colors[op], linewidth=1.5, label=op)

    ax.set_xlabel(obs_labels.get(obs, obs), fontsize=11)
    ax.set_ylabel('BSM / SM', fontsize=10)
    ax.set_ylim(0.3, 2.0)
    ax.legend(fontsize=7, ncol=2)
    ax.set_title(obs_labels.get(obs, obs), fontsize=11)

for idx in range(len(ratio_obs), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig('plots/ratio_panel_key_obs.png', dpi=150, bbox_inches='tight')
plt.show()